This notebook analyzes the importance of parameters for a RAG pipeline correctness.


In [6]:
# This line is needed to be able to import functions from the pr
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
from IPython.display import display, HTML
from plot_results import format_experiment_data, load_statistics


In [7]:
### Here are the hyperparameters of this analysis

# Where the results are stored
data_root = "../data/results"

# The dataset name
dataset_name = "nqopen_small"

# Parameters that are numeric and for which we can compute a correlation with the metrics
numeric_parameters = ['chunk_size','overlap_size','K','top_p','temperature']

In [8]:

# Path to the root of the benchmark results
root_benchmark_path = f'{data_root}/{dataset_name}/parameters/'

assert os.path.exists(root_benchmark_path), f"The path {root_benchmark_path} does not exist. Please make sure that you set correctly the data_root and dataset_name variables."

# Check if a data/ folder already exists
if not os.path.exists(f"{data_root}/{dataset_name}/data"):
    
    # If the data folder does not exis, the benchmark results are still in raw format and we should process them before the analysis
    format_experiment_data(root_benchmark_path)

# Load all statistics in a single dataframe
df_results_all = load_statistics(root_benchmark_path)

# Convert 'Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt' to numeric
df_results_all['Ben'] = pd.to_numeric(df_results_all['Ben'])
df_results_all['Mal'] = pd.to_numeric(df_results_all['Mal'])
df_results_all['Ben&mal'] = pd.to_numeric(df_results_all['Ben&mal'])
df_results_all['Hallucination'] = pd.to_numeric(df_results_all['Hallucination'])
df_results_all['Avg # mal docs in prompt'] = pd.to_numeric(df_results_all['Avg # mal docs in prompt'])

# Set Injection strategy to null if nan
df_results_all.loc[df_results_all['Injection strategy']=='nan','Injection strategy'] = ''

In [9]:
# Select only the Optimization 'unoptimized'
df_results_all = df_results_all[df_results_all['Optimization']=='unoptimized']

The following cell computes the correlation coeficient between the parameters and metrics in the unoptimized case.

In [10]:
correlation_list = []
for param in numeric_parameters:
    parameter_results = df_results_all[df_results_all['Parameter']==param]
    parameter_results['Value'] = pd.to_numeric(parameter_results['Value'])
    parameter_results = parameter_results.sort_values(by='Value')
        
    record = {'parameter':param}
    for m in ['Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt']:
        stats= pd.Series(parameter_results[m])
        correlation = stats.corr(parameter_results['Value'])
        record[f'{m}_correlation'] = round(correlation,3)
        
    correlation_list.append(record)

df_covariance = pd.DataFrame(correlation_list)
display(HTML(df_covariance.to_html(index=True)))

/var/folders/qp/xv46nbnj6wj6rcjlf0g2hlzc0000gn/T/ipykernel_63941/3987545380.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parameter_results['Value'] = pd.to_numeric(parameter_results['Value'])


,parameter,Ben_correlation,Mal_correlation,Ben&mal_correlation,Hallucination_correlation,Avg # mal docs in prompt_correlation
0,chunk_size,0.753,-0.161,-0.487,-0.529,-0.615
1,overlap_size,-0.139,0.189,0.533,-0.521,0.957
2,K,0.990,-0.828,0.663,-0.751,0.983
3,top_p,-0.614,-0.100,0.800,0.205,NaN
4,temperature,-0.975,0.833,0.918,-0.075,NaN
